# 02 – Exploratory Data Analysis & Feature Selection
**Owner:** Member 2 (EDA and Feature Analyst)

**Strict Anti-Leakage Rule:** All exploratory plots, correlations, multicollinearity pruning, and feature selection decisions are performed **exclusively on `data/processed/train.csv`**.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DOCS_DIR, FIGURES_DIR, ID_COL, RAW_NAME_COL, TARGET_COL, TRAIN_PATH
from src.features import SELECTED_FEATURES, add_engineered_features, build_preprocessor, get_xy

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")

train_raw = pd.read_csv(TRAIN_PATH)
train_df = add_engineered_features(train_raw)
print(f"Loaded training split only: {train_df.shape}")

Loaded training split only: (3499, 331)


## Figure 1: Target Distribution (`CO2 uptake at 0.15 bar and 298K`)

In [2]:
fig, ax = plt.subplots(figsize=(8, 4.8))
sns.histplot(train_df[TARGET_COL], bins=45, kde=True, color="#1f77b4", ax=ax)
mean_val = train_df[TARGET_COL].mean()
med_val = train_df[TARGET_COL].median()
ax.axvline(mean_val, color="red", linestyle="--", linewidth=1.8, label=f"Mean = {mean_val:.3f}")
ax.axvline(med_val, color="darkgreen", linestyle=":", linewidth=2.0, label=f"Median = {med_val:.3f}")
ax.set_title("Distribution of CO2 Uptake at 0.15 bar and 298 K (Training Set)")
ax.set_xlabel("CO2 Uptake at 0.15 bar and 298 K (mmol/g assumed)")
ax.set_ylabel("Number of MOFs")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_01_target_distribution.png", dpi=200)
plt.close(fig)
print(train_df[TARGET_COL].describe())
print(f"Skewness: {train_df[TARGET_COL].skew():.3f}")

count    3499.000000
mean        1.781078
std         1.531718
min         0.000027
25%         0.594968
50%         1.377378
75%         2.529592
max         9.318191
Name: CO2 uptake at 0.15 bar and 298K, dtype: float64
Skewness: 1.307


## Figure 2: Distributions of Key Numerical Structural Properties

In [3]:
key_struct = [
    ("ASA [m^2/cm^3]", "Accessible Surface Area (m^2/cm^3)"),
    ("ASA_gravimetric", "Gravimetric ASA (m^2/g)"),
    ("Df", "Pore Limiting Diameter Df (Angstrom assumed)"),
    ("Di", "Largest Cavity Diameter Di (Angstrom assumed)"),
    ("POAV [cm^3/g]", "Accessible Pore Volume POAV (cm^3/g)"),
    ("density [g/cm^3]", "Crystal Density (g/cm^3)"),
]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, (col, label) in zip(axes.ravel(), key_struct):
    sns.histplot(train_df[col], bins=35, kde=True, ax=ax, color="#2a9d8f")
    ax.set_title(f"Distribution of {col}")
    ax.set_xlabel(label)
    ax.set_ylabel("Count")
fig.suptitle("Distributions of Key Structural Properties (Training Set)", y=1.02, fontsize=13)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_02_numerical_distributions.png", dpi=200, bbox_inches="tight")
plt.close(fig)
print("Saved eda_02_numerical_distributions.png")

Saved eda_02_numerical_distributions.png


## Figure 3: Correlation Heatmap (Structural Features, Top RACs & Target)

In [4]:
# Top 6 RAC features most correlated with target in training data
rac_candidates = [c for c in train_df.columns if c not in {ID_COL, RAW_NAME_COL, TARGET_COL} and not c.startswith(("ASA", "D", "NA", "PO", "de", "to", "ca")) or c.startswith(("D_mc", "D_lc", "D_func"))]
non_const_rac = [c for c in rac_candidates if train_df[c].std() > 0]
top_racs = train_df[non_const_rac].corrwith(train_df[TARGET_COL]).abs().sort_values(ascending=False).head(6).index.tolist()

heatmap_cols = [
    "ASA [m^2/cm^3]", "ASA_gravimetric", "Df", "Di", "cage_window_diff",
    "NASA [m^2/cm^3]", "POAV [cm^3/g]", "density [g/cm^3]",
] + top_racs + [TARGET_COL]

corr_mat = train_df[heatmap_cols].corr()
fig, ax = plt.subplots(figsize=(12, 9.5))
sns.heatmap(corr_mat, annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=0.4, ax=ax, annot_kws={"size": 8})
ax.set_title("Pearson Correlation Heatmap: Structural Features, Top RACs, and CO2 Uptake (Train)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_03_correlation_heatmap.png", dpi=200)
plt.close(fig)
print("Top 6 correlated RAC descriptors:", top_racs)

Top 6 correlated RAC descriptors: ['mc_CRY-chi-0-all', 'mc_CRY-chi-2-all', 'mc_CRY-S-0-all', 'sum-mc_CRY-chi-2-all', 'sum-lc-chi-1-all', 'sum-lc-I-0-all']


## Figures 4–8: Physical Relationships with CO₂ Uptake
*Note:* Pressure (0.15 bar) and Temperature (298 K) are fixed operating conditions for every sample in this dataset (there are no varying `pressure` or `temperature` columns), so Figure 6 examines pore volume (`POAV`) and crystal density instead.

In [5]:
# Figure 4: CO2 uptake vs pore size (Df and Di)
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5))
sns.scatterplot(data=train_df, x="Df", y=TARGET_COL, alpha=0.35, s=18, ax=axes[0], color="#264653")
axes[0].set_xlim(0, 25)
axes[0].set_title("CO2 Uptake vs. Pore Limiting Diameter (Df)")
axes[0].set_xlabel("Df (Angstrom assumed)")
axes[0].set_ylabel("CO2 Uptake (mmol/g assumed)")

sns.scatterplot(data=train_df, x="Di", y=TARGET_COL, alpha=0.35, s=18, ax=axes[1], color="#e76f51")
axes[1].set_xlim(0, 30)
axes[1].set_title("CO2 Uptake vs. Largest Cavity Diameter (Di)")
axes[1].set_xlabel("Di (Angstrom assumed)")
axes[1].set_ylabel("CO2 Uptake (mmol/g assumed)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_04_uptake_vs_pore_size.png", dpi=200)
plt.close(fig)

# Figure 5: CO2 uptake vs surface area (volumetric ASA & gravimetric ASA)
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5))
sc1 = axes[0].scatter(train_df["ASA [m^2/cm^3]"], train_df[TARGET_COL], c=train_df["density [g/cm^3]"], cmap="viridis", alpha=0.45, s=18)
axes[0].set_title("CO2 Uptake vs. Volumetric Accessible Surface Area")
axes[0].set_xlabel("ASA [m^2/cm^3]")
axes[0].set_ylabel("CO2 Uptake (mmol/g assumed)")
fig.colorbar(sc1, ax=axes[0], label="Density [g/cm^3]")

sc2 = axes[1].scatter(train_df["ASA_gravimetric"], train_df[TARGET_COL], c=train_df["Di"], cmap="plasma", alpha=0.45, s=18)
axes[1].set_title("CO2 Uptake vs. Gravimetric Accessible Surface Area")
axes[1].set_xlabel("ASA_gravimetric [m^2/g]")
axes[1].set_ylabel("CO2 Uptake (mmol/g assumed)")
fig.colorbar(sc2, ax=axes[1], label="Di (Angstrom assumed)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_05_uptake_vs_surface_area.png", dpi=200)
plt.close(fig)

# Figure 6: CO2 uptake vs Pore Volume (POAV) and Density (fixed 0.15 bar, 298 K)
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5))
sns.scatterplot(data=train_df, x="POAV [cm^3/g]", y=TARGET_COL, alpha=0.35, s=18, ax=axes[0], color="#457b9d")
axes[0].set_title("CO2 Uptake vs. Accessible Pore Volume (0.15 bar, 298 K)")
axes[0].set_xlabel("POAV [cm^3/g]")
axes[0].set_ylabel("CO2 Uptake (mmol/g assumed)")

sns.scatterplot(data=train_df, x="density [g/cm^3]", y=TARGET_COL, alpha=0.35, s=18, ax=axes[1], color="#6d597a")
axes[1].set_title("CO2 Uptake vs. Crystal Density (0.15 bar, 298 K)")
axes[1].set_xlabel("density [g/cm^3]")
axes[1].set_ylabel("CO2 Uptake (mmol/g assumed)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_06_uptake_vs_pore_volume_and_density.png", dpi=200)
plt.close(fig)

# Figure 7: CO2 uptake distribution across Pore-Size Regimes (derived from Df)
pore_bins = [-0.1, 3.3, 5.0, 7.0, 100.0]
pore_labels = [
    "< 3.3 A\n(Narrow / Non-diffusive)",
    "3.3 - 5.0 A\n(Ultra-microporous)",
    "5.0 - 7.0 A\n(Microporous)",
    "> 7.0 A\n(Large Pore)",
]
train_df["pore_regime"] = pd.cut(train_df["Df"], bins=pore_bins, labels=pore_labels)
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=train_df, x="pore_regime", y=TARGET_COL, hue="pore_regime", palette="Set2", legend=False, ax=ax)
ax.set_title("CO2 Uptake by Pore Limiting Diameter (Df) Regime (Training Set)")
ax.set_xlabel("Pore Size Regime (based on Df vs. 3.3 A kinetic diameter of CO2)")
ax.set_ylabel("CO2 Uptake (mmol/g assumed)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_07_uptake_by_pore_regime.png", dpi=200)
plt.close(fig)

# Figure 8: Pair plot of key structural/chemical features and target
pair_cols = ["Di", "Df", "ASA_gravimetric", "density [g/cm^3]", "mc_CRY-chi-0-all", TARGET_COL]
g = sns.pairplot(train_df[pair_cols].sample(n=800, random_state=42), diag_kind="kde", plot_kws={"alpha": 0.35, "s": 14})
g.fig.suptitle("Pair Plot of Key MOF Descriptors and CO2 Uptake (800-Sample Train Subset)", y=1.02)
g.savefig(FIGURES_DIR / "eda_08_pairplot_key_features.png", dpi=160, bbox_inches="tight")
plt.close(g.fig)

print("All 8 EDA figures saved to figures/")
print(train_df.groupby("pore_regime", observed=False)[TARGET_COL].agg(["count", "mean", "median"]))

All 8 EDA figures saved to figures/
                                   count      mean    median
pore_regime                                                 
< 3.3 A\n(Narrow / Non-diffusive)    793  1.746238  1.436208
3.3 - 5.0 A\n(Ultra-microporous)    1502  2.128641  1.822206
5.0 - 7.0 A\n(Microporous)           710  1.556764  0.936458
> 7.0 A\n(Large Pore)                494  1.102636  0.674926


## Verification of `src/features.py` (Zero Leakage & Preprocessing Pipeline)

In [6]:
X_tr, y_tr, id_tr = get_xy("train")
X_va, y_va, id_va = get_xy("val")
X_te, y_te, id_te = get_xy("test")

preprocessor = build_preprocessor()
X_tr_scaled = preprocessor.fit_transform(X_tr)  # Fit ONLY on training data
X_va_scaled = preprocessor.transform(X_va)
X_te_scaled = preprocessor.transform(X_te)

assert X_tr.shape[1] == len(SELECTED_FEATURES) == 193
assert not np.isnan(X_tr_scaled).any() and not np.isnan(X_va_scaled).any() and not np.isnan(X_te_scaled).any()
print(f"Transformed shapes -> Train: {X_tr_scaled.shape}, Val: {X_va_scaled.shape}, Test: {X_te_scaled.shape}")
print("Preprocessing & anti-leakage checks PASSED!")

Transformed shapes -> Train: (3499, 193), Val: (750, 193), Test: (750, 193)
Preprocessing & anti-leakage checks PASSED!
